# otto RAG — **내 파인튜닝 모델**로 돌리는 LangChain RAG + FastAPI + LangSmith

내가 학습시킨 모델(`jjminu/polyglot58-chatbot`, polyglot-ko 5.8B)을 GPU에 올려서:
1. **LangChain RAG** 파이프라인 (검색 → 프롬프트 → 내 모델 → 답변)
2. **FastAPI** REST API (ngrok 터널로 외부 공개)
3. **LangSmith** Tracing + Dataset 평가

> 런타임 유형을 **GPU**(가능하면 A100/L4)로 설정한 뒤 위에서부터 차례로 실행하세요.

## 1. 패키지 설치

In [ ]:
# Chroma 대신 InMemoryVectorStore 사용 -> chromadb 가 numpy 를 다운그레이드하는 충돌 회피
!pip -q install langchain==0.3.13 langchain-community==0.3.13 \
  "langsmith>=0.2.10,<0.3" sentence-transformers pyngrok \
  fastapi "uvicorn[standard]" nest-asyncio langchain-google-genai==2.0.7

## 2. GPU 확인

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (런타임>유형변경에서 GPU 선택)')

## 3. 문서 / 평가셋 파일 생성 (자체 내장)

In [ ]:
import os, json
os.makedirs('docs', exist_ok=True)
docs = {
  "ai_bootcamp_notes.md": "# AI 부트캠프 커리큘럼 안내\n\n## 학습 목표\nAI 트랙은 딥러닝의 기초부터 LLM 애플리케이션 개발까지 다룹니다. 수료 후에는\n이미지 분류 모델과 RAG 기반 챗봇을 직접 만들 수 있는 수준을 목표로 합니다.\n\n## 주차별 내용\n1주차에는 파이썬 복습과 넘파이를 다룹니다. 2주차부터 4주차까지는 신경망과\nCNN을 배우고, ResNet과 VGG16 같은 사전학습 모델로 전이학습을 실습합니다.\n5주차에는 자연어 처리와 챗봇을 만들고, RAG 아키텍처를 적용합니다.\n\n## RAG 실습 내용\nRAG 실습에서는 문서를 청킹하고 sentence-transformers로 임베딩한 뒤 ChromaDB에\n저장합니다. 사용자 질문이 들어오면 코사인 유사도로 관련 문서를 검색해 프롬프트에\n넣고 LLM이 답변을 생성합니다.\n\n## 프로젝트 평가 기준\n최종 프로젝트는 동작 여부, 코드 구조, 그리고 발표로 평가됩니다. 단순히 개념을\n아는 것보다 실제로 동작하는 애플리케이션을 만드는 것이 더 높게 평가됩니다.\n",
  "startupcode_faq.md": "# 스타트업코드(Startupcode) 서비스 안내\n\n## 회사 소개\n스타트업코드는 개발자를 위한 교육과 교재를 제공하는 회사입니다. 주력 서비스로는\n개발 교재 플랫폼 \"어댑터즈(Adapterz)\"와 온라인 코딩 부트캠프가 있습니다.\n2021년에 설립되었으며, 현재 누적 수강생은 1만 2천 명을 넘었습니다.\n\n## 어댑터즈(Adapterz)란\n어댑터즈는 현업 개발자가 직접 집필한 실무 중심 개발 교재를 구독형으로 제공하는\n서비스입니다. 월 구독료는 1만 9천 원이며, 모든 교재를 무제한으로 볼 수 있습니다.\n교재는 매주 새로 업데이트되고, 코드 예제는 깃허브 저장소로 함께 제공됩니다.\n\n## 부트캠프 과정\n온라인 코딩 부트캠프는 16주 과정으로 운영됩니다. 백엔드 트랙과 AI 트랙 두 가지가\n있으며, 백엔드 트랙은 파이썬과 FastAPI를, AI 트랙은 딥러닝과 LLM 애플리케이션\n개발을 다룹니다. 수료하려면 최종 프로젝트를 제출하고 코드 리뷰를 통과해야 합니다.\n\n## 수강료와 환불 정책\n부트캠프 수강료는 350만 원이며, 분할 납부가 가능합니다. 환불은 수강 시작 후\n14일 이내에 신청하면 전액 환불됩니다. 14일이 지난 뒤에는 진행한 주차를 제외한\n잔여 금액의 일부만 환불됩니다.\n\n## 수료증과 취업 지원\n모든 과정을 수료하면 수료증이 발급됩니다. 또한 취업 지원 프로그램을 통해 이력서\n첨삭과 모의 면접을 제공하며, 협력 기업에 추천서를 보내드립니다.\n\n## 문의 방법\n서비스 관련 문의는 평일 오전 10시부터 오후 6시까지 이메일(support@startupcode.kr)\n또는 카카오톡 채널로 받습니다. 주말과 공휴일에는 답변이 다음 영업일로 넘어갑니다.\n"
}
for name, content in docs.items():
    open(f'docs/{name}', 'w', encoding='utf-8').write(content)

dataset_text = "{\"question\": \"어댑터즈 월 구독료는 얼마인가요?\", \"answer\": \"월 1만 9천 원입니다.\", \"source\": \"startupcode_faq.md\"}\n{\"question\": \"부트캠프 수강료는 얼마인가요?\", \"answer\": \"350만 원이며 분할 납부가 가능합니다.\", \"source\": \"startupcode_faq.md\"}\n{\"question\": \"환불은 언제까지 신청하면 전액 환불되나요?\", \"answer\": \"수강 시작 후 14일 이내에 신청하면 전액 환불됩니다.\", \"source\": \"startupcode_faq.md\"}\n{\"question\": \"부트캠프는 몇 주 과정인가요?\", \"answer\": \"16주 과정입니다.\", \"source\": \"startupcode_faq.md\"}\n{\"question\": \"스타트업코드는 몇 년에 설립되었나요?\", \"answer\": \"2021년에 설립되었습니다.\", \"source\": \"startupcode_faq.md\"}\n{\"question\": \"누적 수강생은 몇 명인가요?\", \"answer\": \"1만 2천 명을 넘었습니다.\", \"source\": \"startupcode_faq.md\"}\n{\"question\": \"서비스 문의는 어떤 이메일로 하나요?\", \"answer\": \"support@startupcode.kr 로 문의할 수 있습니다.\", \"source\": \"startupcode_faq.md\"}\n{\"question\": \"AI 트랙 5주차에는 무엇을 배우나요?\", \"answer\": \"자연어 처리와 챗봇을 만들고 RAG 아키텍처를 적용합니다.\", \"source\": \"ai_bootcamp_notes.md\"}\n{\"question\": \"RAG 실습에서 임베딩은 무엇으로 하나요?\", \"answer\": \"sentence-transformers로 임베딩한 뒤 ChromaDB에 저장합니다.\", \"source\": \"ai_bootcamp_notes.md\"}\n{\"question\": \"최종 프로젝트는 무엇으로 평가되나요?\", \"answer\": \"동작 여부, 코드 구조, 발표로 평가됩니다.\", \"source\": \"ai_bootcamp_notes.md\"}\n{\"question\": \"어댑터즈 환불 신청은 카카오톡으로 24시간 가능한가요?\", \"answer\": \"제공된 문서에서 답을 찾을 수 없습니다.\", \"source\": \"\"}\n"
open('dataset.jsonl', 'w', encoding='utf-8').write(dataset_text)
print('문서', len(docs), '개 + dataset.jsonl 생성 완료')

## 4. API 키 설정
- **LangSmith** 키는 필수 (Tracing/평가 기록). https://smith.langchain.com → Settings → API Keys
- **Google** 키는 선택 (평가 correctness judge용). 없으면 source_match만 채점.

In [ ]:
import os, getpass
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'otto-rag-my-model'
os.environ['LANGCHAIN_API_KEY'] = getpass.getpass('LangSmith API Key (lsv2_...): ')
g = getpass.getpass('Google API Key (평가 judge용, 없으면 Enter): ')
if g.strip():
    os.environ['GOOGLE_API_KEY'] = g.strip()

## 5. 내 모델 로드 + 빠른 품질 확인
5.8B 모델이라 다운로드/로드에 몇 분 걸립니다. 아래 생성 결과로 **품질을 먼저 눈으로 확인**하세요.
(토크나이저가 안 잡히면 자동으로 베이스 `polyglot-ko-5.8b` 토크나이저로 폴백합니다.)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'jjminu/polyglot58-chatbot'   # 더 작은 모델: 'jjminu/koalpaca-chatbot'
BASE_TOK = 'EleutherAI/polyglot-ko-5.8b'

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
except Exception as e:
    print('모델 토크나이저 로드 실패 -> 베이스 토크나이저 사용:', str(e)[:80])
    tokenizer = AutoTokenizer.from_pretrained(BASE_TOK)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
print('로드 완료:', round(model.num_parameters()/1e9, 2), 'B params')

# 재학습 모델과 동일하게 greedy 디코딩 + 환각 꼬리 자르기
STOP = ['###', '[출처', '출처:', 'http', '웹사이트', '고객센터', '\n\n', '\n[']

def generate(prompt, max_new_tokens=80):
    ids = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
            repetition_penalty=1.3, no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
    cut = len(text)
    for m in STOP:
        i = text.find(m)
        if i != -1:
            cut = min(cut, i)
    return text[:cut].strip()

print(generate('### 질문: 파이썬이 뭐야?\n\n### 답변:'))

## 6. 임베딩(ko-sroberta) + InMemoryVectorStore 인덱스 구축

In [ ]:
import glob, os, torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

device = 'cuda' if torch.cuda.is_available() else 'cpu'
emb = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask', model_kwargs={'device': device})

documents = []
for p in sorted(glob.glob('docs/*.md')):
    loaded = TextLoader(p, encoding='utf-8').load()
    for d in loaded:
        d.metadata['source'] = os.path.basename(p)
    documents += loaded

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80,
    separators=['\n## ', '\n\n', '\n', '. ', ' ', ''])
chunks = splitter.split_documents(documents)
vs = InMemoryVectorStore.from_documents(chunks, emb)
retriever = vs.as_retriever(search_kwargs={'k': 3})
print('인덱싱 완료:', len(chunks), '청크')

## 7. LCEL RAG 체인 (내 모델 연결)
`retriever | prompt | 내 모델 | 답변` — LangChain의 Runnable 인터페이스로 모델만 갈아끼운 형태.

In [ ]:
from operator import itemgetter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 재파인튜닝 노트북의 학습 프롬프트와 동일해야 효과가 납니다
PROMPT = PromptTemplate.from_template(
    "### 질문: 아래 [문서] 내용만 근거로 질문에 한국어 한 문장으로만 답해줘. "
    "문서에 답이 없으면 다른 말 없이 정확히 '제공된 문서에서 답을 찾을 수 없습니다'라고만 답해. 추측·부연 금지.\n\n"
    '[문서]\n{context}\n\n[질문] {question}\n\n### 답변:'
)

def format_docs(docs):
    return '\n\n'.join(f"[출처: {d.metadata.get('source','?')}] {d.page_content}" for d in docs)

# 내 모델을 LangChain Runnable 로 래핑 (PromptValue -> 문자열 -> 생성)
llm = RunnableLambda(lambda pv: generate(pv.to_string()))

rag_chain = RunnablePassthrough.assign(
    docs=itemgetter('question') | retriever
).assign(
    answer={'context': itemgetter('docs') | RunnableLambda(format_docs),
            'question': itemgetter('question')} | PROMPT | llm
)

def query(q):
    r = rag_chain.invoke({'question': q})
    return {'question': q, 'answer': r['answer'],
            'sources': sorted({d.metadata.get('source','?') for d in r['docs']}),
            'contexts': [d.page_content for d in r['docs']]}

## 8. RAG 동작 테스트

In [ ]:
for q in ['어댑터즈 구독료 얼마야?', '부트캠프 환불 정책 알려줘', '비트코인 사도 돼?']:
    r = query(q)
    print('Q:', q)
    print('A:', r['answer'])
    print('출처:', r['sources'])
    print()

## 9. FastAPI + ngrok 으로 REST API 공개
ngrok 무료 토큰 필요: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import nest_asyncio, threading, time, getpass
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

nest_asyncio.apply()
ngrok.set_auth_token(getpass.getpass('ngrok authtoken: '))

api = FastAPI(title='otto RAG (my model)')

class Query(BaseModel):
    question: str

@api.get('/health')
def health():
    return {'status': 'ok'}

@api.post('/query')
def do_query(req: Query):
    return query(req.question)

ngrok.kill()
public = ngrok.connect(8000)
print('PUBLIC URL:', public.public_url)
threading.Thread(target=lambda: uvicorn.run(api, host='0.0.0.0', port=8000), daemon=True).start()
time.sleep(3)
print('서버 기동 완료')

## 10. API 호출 테스트

In [ ]:
import requests
print(requests.get(public.public_url + '/health').json())
r = requests.post(public.public_url + '/query', json={'question': '부트캠프 수강료 얼마야?'})
print(r.json())

## 11. LangSmith Dataset 평가
- **correctness**: Gemini judge (Google 키 있을 때만, 내 모델 답변을 채점)
- **source_match**: 기대 출처가 검색 결과에 포함됐는지 (LLM 불필요)

In [ ]:
import json, os
from langsmith import Client, evaluate

client = Client()
data = [json.loads(l) for l in open('dataset.jsonl', encoding='utf-8') if l.strip()]
NAME = 'otto-rag-qa-my-model'

if not client.has_dataset(dataset_name=NAME):
    ds = client.create_dataset(dataset_name=NAME)
    client.create_examples(
        inputs=[{'question': r['question']} for r in data],
        outputs=[{'answer': r['answer'], 'source': r['source']} for r in data],
        dataset_id=ds.id)
    print('Dataset 업로드:', len(data))

def target(inputs):
    return query(inputs['question'])

def correctness(run, example):
    if not os.environ.get('GOOGLE_API_KEY'):
        return {'key': 'correctness', 'score': None}
    from langchain_google_genai import ChatGoogleGenerativeAI
    judge = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite', temperature=0)
    pred = (run.outputs or {}).get('answer', '')
    ref = (example.outputs or {}).get('answer', '')
    v = judge.invoke('기준정답과 답변이 핵심 사실에서 일치하면 CORRECT, 아니면 INCORRECT 만 답해.\n'
                     f'기준: {ref}\n답변: {pred}').content
    return {'key': 'correctness', 'score': int('CORRECT' in v.upper() and 'INCORRECT' not in v.upper())}

def source_match(run, example):
    exp = (example.outputs or {}).get('source', '')
    src = (run.outputs or {}).get('sources', [])
    if not exp:
        return {'key': 'source_match', 'score': None}
    return {'key': 'source_match', 'score': int(exp in src)}

results = evaluate(target, data=NAME, evaluators=[correctness, source_match],
    experiment_prefix='otto-my-model', max_concurrency=1)
print('평가 완료 — https://smith.langchain.com 에서 확인')

---
**확인:** https://smith.langchain.com → Projects `otto-rag-my-model`(Tracing), Datasets & Experiments `otto-rag-qa-my-model`(평가 점수).

내 모델 답변 품질이 낮으면, 데이터/에폭을 늘려 재파인튜닝 후 같은 노트북에서 `MODEL_ID` 만 바꾸면 됩니다.